# 예선 — XGBoost + CatBoost 앙상블

`01_eda_and_feature_engineering.ipynb` 에서 저장한 처리된 셋 위에서 두 개의 파이프라인을 학습하고 평균(0.5 : 0.5) 으로 블렌딩합니다.

| 파이프라인 | 모델 | split 기준 | 핵심 피처 |
| --- | --- | --- | --- |
| Pipeline A | XGBoost | `weather_mean` 결측 여부 | 기본 + 시간 + 기상 |
| Pipeline B | CatBoost | `weather_mean` 결측 여부 + `DIST > 0` | A 피처 + target encoding |

각 파이프라인은 5-fold KFold 로 학습하며, 폴드 평균 OOF MAE 와 테스트 예측을 반환합니다.


## Imports

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import KFold

from xgboost import XGBRegressor
from catboost import CatBoostRegressor

warnings.filterwarnings("ignore")
SEED = 42
N_SPLITS = 5

DATA_DIR = Path("data")
PROCESSED_DIR = DATA_DIR / "processed" / "preliminaries"
SUBMISSION_DIR = Path("submissions")
SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)

## 처리된 데이터 로드

In [ ]:
train = pd.read_parquet(PROCESSED_DIR / "train.parquet")
test = pd.read_parquet(PROCESSED_DIR / "test.parquet")
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv")

print(f"train: {train.shape}, test: {test.shape}")

## 공통 학습 함수

In [ ]:
def run_kfold(model_factory, X: pd.DataFrame, y: pd.Series, X_test: pd.DataFrame, n_splits: int = N_SPLITS, seed: int = SEED) -> tuple[np.ndarray, list[float]]:
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    test_pred = np.zeros(len(X_test))
    fold_scores = []

    for fold_index, (train_idx, valid_idx) in enumerate(kf.split(X), start=1):
        X_tr, X_va = X.iloc[train_idx], X.iloc[valid_idx]
        y_tr, y_va = y.iloc[train_idx], y.iloc[valid_idx]

        model = model_factory()
        model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
        valid_pred = np.clip(model.predict(X_va), 0, None)
        score = mean_absolute_error(y_va, valid_pred)
        fold_scores.append(score)
        print(f"  fold {fold_index}: MAE={score:.4f}")

        test_pred += np.clip(model.predict(X_test), 0, None) / n_splits

    print(f"  → mean MAE={np.mean(fold_scores):.4f} (std={np.std(fold_scores):.4f})")
    return test_pred, fold_scores

## Pipeline A — XGBoost (`weather_mean` split)

In [ ]:
def split_by_weather(frame: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    has_weather = frame[frame["weather_mean"].notna()].reset_index()
    no_weather = frame[frame["weather_mean"].isna()].reset_index()
    return has_weather, no_weather


train_with, train_without = split_by_weather(train)
test_with, test_without = split_by_weather(test)

DROP_WITH = ["ID", "FLAG", "year", "day", "hour", "minute", "CO_PO_mean", "CO_PO_ym_mean", "CO_PO_SH_y_mean"]
DROP_WITHOUT = [
    "ID", "BREADTH", "BUILT", "DEADWEIGHT", "DEPTH", "DRAUGHT", "SHIPMANAGER", "FLAG",
    "U_WIND", "V_WIND", "AIR_TEMPERATURE", "BN", "ATA_LT", "month", "day", "hour", "minute",
    "CO_PO_mean", "CO_PO_ym_mean", "CO_PO_SH_y_mean",
]


def slice_xy(frame: pd.DataFrame, drop: list[str]) -> tuple[pd.DataFrame, pd.Series, pd.Index]:
    drop_cols = [c for c in drop if c in frame.columns] + ["index"]
    y = frame["CI_HOUR"]
    X = frame.drop(columns=[c for c in drop_cols + ["CI_HOUR"] if c in frame.columns])
    return X, y, frame["index"]


X_with, y_with, idx_with_train = slice_xy(train_with, DROP_WITH)
X_without, y_without, idx_without_train = slice_xy(train_without, DROP_WITHOUT)
X_test_with = test_with.drop(columns=[c for c in DROP_WITH + ["index"] if c in test_with.columns])
X_test_without = test_without.drop(columns=[c for c in DROP_WITHOUT + ["index"] if c in test_without.columns])
idx_with_test, idx_without_test = test_with["index"], test_without["index"]

In [ ]:
def xgb_factory_with():
    return XGBRegressor(
        objective="reg:absoluteerror", eta=0.01, n_estimators=2000,
        max_depth=11, min_child_weight=15, max_delta_step=15, colsample_bytree=1.0,
        reg_alpha=0, reg_lambda=1, gamma=0, max_bin=2048,
        eval_metric="mae", early_stopping_rounds=200, n_jobs=-1, seed=SEED, verbosity=0,
    )


def xgb_factory_without():
    return XGBRegressor(
        objective="reg:absoluteerror", eta=0.01, n_estimators=2000,
        max_depth=7, min_child_weight=10, max_delta_step=2.2, colsample_bytree=0.8,
        reg_alpha=4, reg_lambda=7, gamma=0, max_bin=64,
        eval_metric="mae", early_stopping_rounds=200, n_jobs=-1, seed=SEED, verbosity=0,
    )


print("[XGB] weather_mean 있음")
xgb_pred_with, _ = run_kfold(xgb_factory_with, X_with, y_with, X_test_with)
print("[XGB] weather_mean 없음")
xgb_pred_without, _ = run_kfold(xgb_factory_without, X_without, y_without, X_test_without)

submission_a = sample_submission.copy()
submission_a.loc[idx_with_test, "CI_HOUR"] = xgb_pred_with
submission_a.loc[idx_without_test, "CI_HOUR"] = xgb_pred_without

## Pipeline B — CatBoost (`DIST > 0` 추가 split)

In [ ]:
train_b_with = train_with[train_with["DIST"] > 0].reset_index(drop=True)
train_b_without = train_without
test_b_with = test_with
test_b_without = test_without

DROP_B_WITH = ["ID", "day", "hour", "minute", "ARI_CO", "ARI_PO", "BREADTH", "weekday"]
DROP_B_WITHOUT = DROP_WITHOUT


def slice_drop(frame: pd.DataFrame, drop: list[str]) -> pd.DataFrame:
    return frame.drop(columns=[c for c in drop if c in frame.columns])


X_b_with = slice_drop(train_b_with, DROP_B_WITH + ["CI_HOUR", "index"])
y_b_with = train_b_with["CI_HOUR"]
X_b_without = slice_drop(train_b_without, DROP_B_WITHOUT + ["CI_HOUR", "index"])
y_b_without = train_b_without["CI_HOUR"]
X_test_b_with = slice_drop(test_b_with, DROP_B_WITH + ["index"])
X_test_b_without = slice_drop(test_b_without, DROP_B_WITHOUT + ["index"])


def cat_factory():
    return CatBoostRegressor(
        n_estimators=20000, learning_rate=0.03, depth=8,
        eval_metric="MAE", loss_function="MAE",
        early_stopping_rounds=200, random_state=SEED, verbose=False,
    )


print("[CatBoost] DIST>0 + weather_mean 있음")
cat_pred_with, _ = run_kfold(cat_factory, X_b_with, y_b_with, X_test_b_with)
print("[CatBoost] weather_mean 없음")
cat_pred_without, _ = run_kfold(cat_factory, X_b_without, y_b_without, X_test_b_without)

submission_b = sample_submission.copy()
submission_b.loc[idx_with_test, "CI_HOUR"] = cat_pred_with
submission_b.loc[idx_without_test, "CI_HOUR"] = cat_pred_without

# DIST==0 행은 항해 거리가 없는 케이스로 대기시간 0 으로 후처리
zero_dist_idx = test_b_with.loc[test_b_with["DIST"] == 0, "index"]
submission_b.loc[zero_dist_idx, "CI_HOUR"] = 0

## 최종 블렌딩 제출

In [ ]:
final_submission = sample_submission.copy()
final_submission["CI_HOUR"] = 0.5 * submission_a["CI_HOUR"] + 0.5 * submission_b["CI_HOUR"]
final_submission["CI_HOUR"] = final_submission["CI_HOUR"].clip(lower=0)

output_path = SUBMISSION_DIR / "preliminaries_xgb_catboost_blend.csv"
final_submission.to_csv(output_path, index=False)
print(f"saved: {output_path}")
final_submission.head()

## 정리

- 두 파이프라인 모두 5-fold KFold + early stopping + 폴드별 MAE 평균을 기록했습니다.
- 데이터를 `weather_mean` 결측 여부로 분기해 각 분기에 적합한 피처 셋과 모델 하이퍼파라미터를 사용했습니다.
- CatBoost 파이프라인은 `DIST > 0` 추가 분기와 target encoding 을 활용해 항해 거리가 있는 케이스를 더 정밀하게 학습했습니다.
- 0.5/0.5 블렌딩 후 음수 클리핑으로 최종 제출을 만들었습니다.
